https://lweitkamp.github.io/triton_exercises/introduction/block_pointers.html

In [12]:
import torch
world_size = 4
outputs = torch.zeros((1,5), dtype=torch.float32)
input = torch.randn(world_size,5, dtype=torch.float32)
print(f"Output : {outputs.shape[0] * world_size} : Input : {input.shape[0]}")

Output : 4 : Input : 4


In [ ]:
import pytest
import torch
import os

import triton


def _attn_fwd_inner(acc, l_i, m_i, q,  #
                    desc_k, desc_v,  #
                    offset_y, dtype, start_m, qk_scale,  #
                    BLOCK_M, HEAD_DIM, BLOCK_N,  #
                    STAGE, offs_m, offs_n,  #
                    N_CTX, warp_specialize):
    # range of values handled by this stage
    print(STAGE)
    if STAGE == 1:
        lo, hi = 0, start_m * BLOCK_M
    elif STAGE == 2:
        lo, hi = start_m * BLOCK_M, (start_m + 1) * BLOCK_M
        # lo = tl.multiple_of(lo, BLOCK_M)
    # causal = False
    else:
        lo, hi = 0, N_CTX
    offsetk_y = offset_y + lo

    #dtype == tl.float8e5
    if dtype == torch.float8_e5m2:
        offsetv_y = offset_y * HEAD_DIM + lo
    else:
        offsetv_y = offset_y + lo

    # loop over k, v and update accumulator
    # TODO what is warp_specialize in tl.range ?
    # for start_n in tl.range(lo, hi, BLOCK_N, warp_specialize=warp_specialize):
    for start_n in torch.arange(lo, hi, BLOCK_N):
        # start_n = tl.multiple_of(start_n, BLOCK_N)
        # -- compute qk ----
        # k = desc_k.load([offsetk_y, 0]).T
        # qk = tl.dot(q, k)
        if STAGE == 2:
            mask = offs_m[:, None] >= (start_n + offs_n[None, :])
            # print(f"mask is used {mask}")
            # qk = qk * qk_scale + tl.where(mask, 0, -1.0e6)
            # m_ij = tl.maximum(m_i, tl.max(qk, 1))
            # qk -= m_ij[:, None]
        else:
            print("no mask used")
            # m_ij = tl.maximum(m_i, tl.max(qk, 1) * qk_scale)
            # qk = qk * qk_scale - m_ij[:, None]
        # p = tl.math.exp2(qk)
        # -- compute correction factor
        # alpha = tl.math.exp2(m_i - m_ij)
        # l_ij = tl.sum(p, 1)
        # -- update output accumulator --
        # if not IS_HOPPER and warp_specialize and BLOCK_M == 128 and HEAD_DIM == 128:
        #     BM: tl.constexpr = acc.shape[0]
        #     BN: tl.constexpr = acc.shape[1]
        #     acc0, acc1 = acc.reshape([BM, 2, BN // 2]).permute(0, 2, 1).split()
        #     acc0 = acc0 * alpha[:, None]
        #     acc1 = acc1 * alpha[:, None]
        #     acc = tl.join(acc0, acc1).permute(0, 2, 1).reshape([BM, BN])
        # else:
        #     acc = acc * alpha[:, None]
        # prepare p and v for the dot
        # if dtype == tl.float8e5:
        #     v = desc_v.load([0, offsetv_y]).T
        # else:
        #     v = desc_v.load([offsetv_y, 0])
        # p = p.to(dtype)
        # note that this non transposed v for FP8 is only supported on Blackwell
        # acc = tl.dot(p, v, acc)
        # update m_i and l_i
        # place this at the end of the loop to reduce register pressure
    #     l_i = l_i * alpha + l_ij
    #     m_i = m_ij
    #     offsetk_y += BLOCK_N
    #     offsetv_y += BLOCK_N
    # return acc, l_i, m_i
    print("----inner-----")

def _attn_fwd(sm_scale, M, Z, H, desc_q, desc_k, desc_v, desc_o,
              HEAD_DIM,  #
              BLOCK_M,  #
              BLOCK_N,  #
              FP8_OUTPUT,  #
              STAGE,  #
              warp_specialize, N_CTX):
  dtype = torch.float8_e5m2 if FP8_OUTPUT else torch.float16
  # TODO
  # start_m = tl.program_id(0)
  start_m = 0
  # TODO
  # off_hz = tl.program_id(1)
  off_hz = 100
  for start_m in range(2):
    for off_hz in range(100,148):
      print(f"start_m : {start_m}, off_hz : {off_hz}")
      off_z = off_hz // H
      off_h = off_hz % H
      # print(f"off_z : {off_z},off_h : {off_h}")
      y_dim = Z * H * N_CTX
      # print(f"y_dim : {y_dim}")
      offset_y = off_z * (N_CTX * H) + off_h * N_CTX
      # print(f"offset_y : {offset_y}")
      qo_offset_y = offset_y + start_m * BLOCK_M
      # print(f"qo_offset_y : {qo_offset_y}")
      # initialize offsets
      offs_m = start_m * BLOCK_M + torch.arange(0, BLOCK_M)
      offs_n = torch.arange(0, BLOCK_N)
      # print(f"offs_m : {offs_m}")
      # print(f"offs_n : {offs_n}")
      # initialize pointer to m and l
      m_i = torch.zeros([BLOCK_M], dtype=torch.float32) - float("inf")
      l_i = torch.zeros([BLOCK_M], dtype=torch.float32) + 1.0
      acc = torch.zeros([BLOCK_M, HEAD_DIM], dtype=torch.float32)
      # load scales
      qk_scale = sm_scale
      qk_scale *= 1.44269504  # 1/log(2)
      # q = desc_q.load([qo_offset_y, 0])
      print(f"q load : {[qo_offset_y, 0]}")
      q = torch.randn([qo_offset_y, 0], dtype=dtype, device=DEVICE, requires_grad=True)
      # print(f"q : {q}")
      if STAGE & 1:
                  _attn_fwd_inner(acc, l_i, m_i, q,  #
                                  desc_k, desc_v,  #
                                  offset_y, dtype, start_m, qk_scale,  #
                                  BLOCK_M, HEAD_DIM, BLOCK_N,  #
                                  4 - STAGE, offs_m, offs_n, N_CTX,  #
                                  warp_specialize)
      # stage 2: on-band
      if STAGE & 2:
                  _attn_fwd_inner(acc, l_i, m_i, q,  #
                                  desc_k, desc_v,  #
                                  offset_y, dtype, start_m, qk_scale,  #
                                  BLOCK_M, HEAD_DIM, BLOCK_N,  #
                                  2, offs_m, offs_n, N_CTX,  #
                                  warp_specialize)
      # epilogue
      # m_i += tl.math.log2(l_i)
      # acc = acc / l_i[:, None]
      # m_ptrs = M + off_hz * N_CTX + offs_m
      # tl.store(m_ptrs, m_i)
      # desc_o.store([qo_offset_y, 0], acc.to(dtype))
      print("----------")


def attention(q, k, v, causal, sm_scale, warp_specialize=True):
  # shape constraints
  HEAD_DIM_Q, HEAD_DIM_K = q.shape[-1], k.shape[-1]
  # when v is in float8_e5m2 it is transposed.
  HEAD_DIM_V = v.shape[-1]
  o = torch.empty_like(q)
  stage = 3 if causal else 1
  M = torch.empty((q.shape[0], q.shape[1], q.shape[2]), device=q.device, dtype=torch.float32)
  desc_q = q
  desc_v = v
  desc_k = k
  desc_o = o
  BLOCK_M = 64 #128
  BLOCK_N = 32 # 64 #128
  grid = (q.shape[2]// BLOCK_M, q.shape[0] * q.shape[1], 1)
  print(f"grid : {grid}")
  _attn_fwd(sm_scale, M, q.shape[0], q.shape[1],  #
            desc_q, desc_k, desc_v, desc_o,  #
            N_CTX=q.shape[2],  #
            HEAD_DIM=HEAD_DIM_K,  #
            BLOCK_M=BLOCK_M,  #
            BLOCK_N=BLOCK_N,  #
            FP8_OUTPUT=q.dtype == torch.float8_e5m2,  #
            STAGE=stage,  #
            warp_specialize=warp_specialize)

# import triton.language as tl
# from triton.tools.tensor_descriptor import TensorDescriptor
DEVICE = triton.runtime.driver.active.get_active_torch_device()
sm_scale = 0.5
causal = True
warp_specialize = False #True
BATCH = 1 #4
H = 48 #2
N_CTX = 128 #1024
HEAD_DIM = 64 #64
dtype = torch.float16
q = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)
k = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)
v = torch.randn((BATCH, H, N_CTX, HEAD_DIM), dtype=dtype, device=DEVICE, requires_grad=True)

q = q.to(torch.float8_e5m2)
k = k.to(torch.float8_e5m2)
v = v.permute(0, 1, 3, 2).contiguous()
v = v.permute(0, 1, 3, 2)
v = v.to(torch.float8_e5m2)
print(f"V Shape : {v.shape}")

attention(q, k ,v, causal, sm_scale, warp_specialize)




Zero
2
